In [4]:
from src.tokenizers.BPE import BytePairEncoding
from src.train.lit_transformer import LitTransformer
from src.models.transformer import Transformer

# You need to instantiate the model used by LitTransformer and pass it in as the argument.
# Assume you know (or can specify) the Transformer architecture / args used at training.
# Here is an example (adjust paths/params as needed):

# Example: Load model config -- in production, load those from the checkpoint or your experiment config!
vocab_size = 37000          # Use actual vocab_size used for training
d_model = 256               # actual d_model
d_ff = 512                  # actual d_ff
n_heads = 8                 # actual n_heads
N = 6                       # actual N
seq_len = 256               # actual seq_len

# Create the underlying Transformer model -- adjust the imports and values as needed
transformer = Transformer(
    vocab_size=vocab_size,
    d_model=d_model,
    d_ff=d_ff,
    n_heads=n_heads,
    N=N,
    seq_len=seq_len,
)

# Now load the LitTransformer with the transformer argument
lit_model = LitTransformer.load_from_checkpoint(
    "/home/mcvjetko/phd/projects/transformer/experiments/en-hr/last.ckpt",
    transformer=transformer
)
transformer = lit_model.transformer
transformer.cuda()
bpe_tokenizer = BytePairEncoding.from_file("/home/mcvjetko/phd/projects/transformer/experiments/en-hr-tokenizers/tokenizer_37000.json")

In [5]:
print(bpe_tokenizer._special_vocab)
print(bpe_tokenizer.inv_vocab[4])
print(bpe_tokenizer.inv_vocab[5])
print(bpe_tokenizer.inv_vocab[6])
print(bpe_tokenizer.inv_vocab[20257])
print(bpe_tokenizer.inv_vocab[582])
print(bpe_tokenizer.inv_vocab[661])
print(bpe_tokenizer.inv_vocab[19172])

{'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '<UNK>': 3}
 
!
"
studying
is
ly
Les


In [38]:
import torch

raw = "Lesly eats lunch with Marko."
text = bpe_tokenizer.tokenize(raw, max_length=256, add_special=True, pad=True)
#print("decoded", bpe_tokenizer.decode(text))
text = torch.tensor(text).to("cuda").unsqueeze(0)
#print(text)
# print(text.shape)
y = torch.full((1, 1), 1, dtype=torch.long).to("cuda")
# print(y.shape)
# print(y)
print(raw + " -> " + bpe_tokenizer.decode(transformer.translate(text, y)[0].tolist()))


Lesly eats lunch with Marko. -> - Les s Les Lese s rukom.
